### Imports

In [ ]:
import logging

from kcatbench.dataset import brenda_build_db, brenda_filter_db, ee_build_db
from kcatbench.util import DATA_DIR

### BRENDA

#### Logging configuration

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    force=True,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

#### Build dataset
Builds a brenda dataset using the BRENDA txt flatfile which needs to be downloaded from BRENDA manually.

The enzyme sequence will be automatically acquired from Uniprot. The substrates and products names are tried to be converted to SMILES representation using a combination of BRENDA ligand mappings (using the BRENDA ligand dataset which also needs to be manually downloaded by placing a blank query) and the MoleculeResolver package, which queries multiple databases.

This process takes a very long time (~12 hours).

The resulting dataset is automatically saved in the brenda data directory.

In [ ]:
df = brenda_build_db(
    flatfile_path=(DATA_DIR / "brenda" / "brenda_2026_1.txt"),
    write_artifacts=True,
    require_uniprot=True,
    enrich_sequences=True,
    enable_logging=True,
    log_every_n_records=10000,
)

#### Filter dataset

Filters the created dataset in these steps:
1. Keep rows with non-missing sequence.
2. Keep rows with non-missing experimental_kcat.
3. Keep rows where kcat_substrate_name appears in substrates_names.
4. Keep rows where the first substrates (SMILES) item is not None/NaN.
5. Remove None/NaN values from substrates (SMILES) and products (SMILES) lists only.

In [ ]:
csv_path, pkl_path = brenda_filter_db(input_df=df)

### EnzyExtract

Downloads the EnzyExtract dataset and filters for wildtype data only. Canonicalizes all substrate SMILES and brings the dataset into a form so it can be used for prediction.

In [ ]:
csv_path, pkl_path = ee_build_db()